# Lesson 11 — I2C 地址、ACK 和设备缺失 — 参考实现

先完成主 Notebook 的练习再阅读。此 Notebook 只展示可通过 exercise 契约的一种实现。


In [ ]:
# 运行支架：只运行一次，不需要学习或修改 Python。
from pathlib import Path
project = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "tools/notebook_setup.py").is_file()), None)
if project is None:
    raise RuntimeError("请用 VS Code 打开 esp32-hands-on-academy 文件夹")
get_ipython().run_line_magic("run", str(project / "tools/notebook_setup.py"))


运行代码格，使用同一个可交互面板验证。然后回到主 Notebook，对照自己的状态和边界处理。


In [ ]:
%%academy_lab 11-i2c-sensor solution exercise pass
#pragma once
#include "academy/device.hpp"
namespace academy {
    class Controller {
    public:
        explicit Controller(Device&);
        void tick();

    private:
        Device& board_;
    };
}

#include <string>
namespace academy {
    Controller::Controller(Device& b) : board_(b) {
    }
    void Controller::tick() {
        board_.output.error = false;
        if (!board_.input.connected || board_.input.fault) {
            board_.output.error = true;
            board_.output.outgoing = "TIMEOUT\n";
            board_.output.state = "i2c error";
            return;
        }
        if (board_.input.text == "I2C:0x48:READ") {
            board_.output.reading = board_.input.temperature;
            board_.output.outgoing =
                "ACK:" + std::to_string(board_.output.reading) + "\n";
            board_.output.state = "i2c ok";
        } else if (!board_.input.text.empty()) {
            board_.output.error = true;
            board_.output.outgoing = "NACK\n";
            board_.output.state = "i2c error";
        }
    }
}
